# Day 47 · 3 — Iceberg SCD Type 2 and validation

Run after notebook 2. New product CDC events are applied in **source sequence order**, not
collapsed to the latest product row. This preserves several changes arriving in one batch.
All SQL is visible. This small lab processes one event at a time; it is intentionally not a
production streaming framework. Run only one instance of this notebook for a DB.
Use the same `DB` and warehouse as notebook 2, and a Spark 3.5 kernel with a compatible Iceberg runtime.

In [ ]:
import os
import re
from pathlib import Path

DB = "cdc_scd_db"  # Use the same database name in all three notebooks.
assert re.fullmatch(r"[a-zA-Z][a-zA-Z0-9_]{0,63}", DB)
LAB_DIR = Path.cwd() / "lab_data" / DB
LAB_DIR.mkdir(parents=True, exist_ok=True)
BRONZE = f"hdfs:///bronze/{DB}"
WAREHOUSE = f"hdfs:///warehouse/{DB}"
print("MySQL database:", DB)

In [ ]:
import sys
import subprocess

# Use the Spark, Java and Hadoop installation configured for your notebook kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
import pyspark
assert pyspark.__version__.startswith("3.5."), "Use a Spark 3.5 kernel with a matching Iceberg runtime."
from pyspark.sql import SparkSession, Window, functions as F

active = SparkSession.getActiveSession()
if active is not None:
    assert active.conf.get("spark.sql.catalog.lab.warehouse", "") == WAREHOUSE, "Restart the kernel when switching DB or Spark configuration."
    assert "IcebergSparkSessionExtensions" in active.conf.get("spark.sql.extensions", ""), "Restart the kernel to enable Iceberg."

spark = (SparkSession.builder.master("local[2]").appName(DB)
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.lab", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lab.type", "hadoop")
    .config("spark.sql.catalog.lab.warehouse", WAREHOUSE)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.sql("CREATE NAMESPACE IF NOT EXISTS lab.ecommerce")
print("Spark:", spark.version)

## History table and a tiny applied-event ledger

`product_sk = cdc_id` identifies a version deterministically. Track name, category, color and
list price; changing only `updated_at` creates no version. A DELETE closes the current
version without inserting a tombstone. A later INSERT starts a new lifetime.

Validity intervals are half-open: **[valid_from, valid_to)**. NULL valid_to means current.
The ledger prevents replaying completed events; deterministic keys also handle retrying an
event interrupted between expiration, insertion and recording completion. There is no
cross-table transaction: do not read partially processed results or run concurrent writers.

In [ ]:
assert spark.catalog.tableExists("lab.ecommerce.product_changes"), "Run notebook 2 first."
spark.sql("""CREATE TABLE IF NOT EXISTS lab.ecommerce.dim_product_scd2 (
    product_sk STRING, product_id BIGINT, product_name STRING, category STRING, color STRING,
    list_price DECIMAL(10,2), valid_from TIMESTAMP, valid_to TIMESTAMP, is_current BOOLEAN)
    USING iceberg""")
spark.sql("""CREATE TABLE IF NOT EXISTS lab.ecommerce.scd2_applied_events (
    cdc_id STRING, sequence BIGINT, batch_id BIGINT) USING iceberg""")
pending = spark.sql("""SELECT s.* FROM lab.ecommerce.product_changes s
    LEFT ANTI JOIN lab.ecommerce.scd2_applied_events p ON s.cdc_id=p.cdc_id
    ORDER BY s.sequence""")
pending.show(100, truncate=False)

## Expire the previous version, then insert the new version

`<=>` is Spark SQL's null-safe equality: two NULL categories count as unchanged.
For each INSERT/UPDATE, compare tracked attributes to the current row. If unchanged, skip
both history writes. If changed, expire then insert. A retry after expiration still sees
the new version is missing; a retry after insertion sees identical current attributes.
For DELETE, simply expire. Record completion only after the history writes succeed.

In [ ]:
source_schema = pending.schema
for change in pending.collect():  # Tiny training dataset; not for large production CDC.
    spark.createDataFrame([change], source_schema).createOrReplaceTempView("one_change")
    if change.operation == "D":
        spark.sql("""MERGE INTO lab.ecommerce.dim_product_scd2 t USING one_change s
            ON t.product_id=s.product_id AND t.is_current=true
            WHEN MATCHED THEN UPDATE SET t.valid_to=s.event_time, t.is_current=false""")
    else:
        changed = spark.sql("""SELECT s.* FROM one_change s
            LEFT JOIN lab.ecommerce.dim_product_scd2 t ON s.product_id=t.product_id AND t.is_current=true
            WHERE t.product_sk IS NULL OR NOT (
                t.product_name <=> s.product_name AND t.category <=> s.category
                AND t.color <=> s.color AND t.list_price <=> s.list_price)""").count() > 0
        if changed:
            spark.sql("""MERGE INTO lab.ecommerce.dim_product_scd2 t USING one_change s
                ON t.product_id=s.product_id AND t.is_current=true AND t.product_sk<>s.cdc_id
                WHEN MATCHED THEN UPDATE SET t.valid_to=s.event_time, t.is_current=false""")
            spark.sql("""MERGE INTO lab.ecommerce.dim_product_scd2 t USING one_change s
                ON t.product_sk=s.cdc_id
                WHEN NOT MATCHED THEN INSERT
                    (product_sk,product_id,product_name,category,color,list_price,valid_from,valid_to,is_current)
                    VALUES(s.cdc_id,s.product_id,s.product_name,s.category,s.color,s.list_price,s.event_time,NULL,true)""")
    spark.sql("""MERGE INTO lab.ecommerce.scd2_applied_events t USING one_change s
        ON t.cdc_id=s.cdc_id WHEN NOT MATCHED THEN INSERT (cdc_id,sequence,batch_id)
        VALUES(s.cdc_id,s.sequence,s.batch_id)""")
    print("Applied", change.sequence, change.operation, "product", change.product_id)

## Compare current state with preserved history

After Change 4, product 1 has **one SCD1 row** and **five SCD2 versions**:
Premium/Black/100 → Premium/Black/95 → Premium/Silver/95 → Standard/Silver/95 → Standard/Silver/90.
Change 5 must leave those five versions unchanged. Change 6 expires product 3; Change 7
creates a second, current lifetime. Source CDC timestamps determine validity boundaries.

In [ ]:
spark.table("lab.ecommerce.dim_product_scd1").orderBy("product_id").show(truncate=False)
spark.table("lab.ecommerce.dim_product_scd2").orderBy("product_id", "valid_from").show(100, truncate=False)
spark.table("lab.ecommerce.latest_orders").orderBy("order_id").show(truncate=False)

## Checks that hold after every change

Verify unique versions, at most one current row, valid nonoverlapping intervals, agreement
between current SCD1/SCD2 attributes and complete event processing. Rerun notebooks 2 and 3:
counts and history should not change. These assertions also work before the final change.

In [ ]:
assert spark.sql("""SELECT product_sk FROM lab.ecommerce.dim_product_scd2
    GROUP BY product_sk HAVING COUNT(*)>1""").count() == 0
assert spark.sql("""SELECT product_id FROM lab.ecommerce.dim_product_scd2 WHERE is_current
    GROUP BY product_id HAVING COUNT(*)>1""").count() == 0
assert spark.sql("""SELECT * FROM lab.ecommerce.dim_product_scd2
    WHERE (is_current AND valid_to IS NOT NULL) OR (NOT is_current AND valid_to IS NULL)
       OR valid_to <= valid_from""").count() == 0
assert spark.sql("""SELECT * FROM (
    SELECT *, LEAD(valid_from) OVER(PARTITION BY product_id ORDER BY valid_from) AS next_from
    FROM lab.ecommerce.dim_product_scd2)
    WHERE next_from IS NOT NULL AND (valid_to IS NULL OR valid_to > next_from)""").count() == 0
attrs = ["product_id", "product_name", "category", "color", "list_price"]
scd1 = spark.table("lab.ecommerce.dim_product_scd1").select(*attrs)
scd2 = spark.table("lab.ecommerce.dim_product_scd2").filter("is_current").select(*attrs)
assert scd1.exceptAll(scd2).count() == 0 and scd2.exceptAll(scd1).count() == 0
assert spark.sql("""SELECT c.cdc_id FROM lab.ecommerce.product_changes c
    LEFT ANTI JOIN lab.ecommerce.scd2_applied_events p ON c.cdc_id=p.cdc_id""").count() == 0
print("PASS: current state, version uniqueness, interval validity and CDC coverage")

## Final scenario checks (after Change 4 or later)

This cell waits until the final price change has arrived. Late PAID must not replace
DELIVERED for order 1001. Order 1006 checks the equal-business-time tie-break.

In [ ]:
final = spark.table("lab.ecommerce.dim_product_scd1").filter("product_id=1 AND list_price=90").count() == 1
if final:
    versions = spark.sql("""SELECT category,color,list_price FROM lab.ecommerce.dim_product_scd2
        WHERE product_id=1 ORDER BY valid_from""").collect()
    assert [(r.category,r.color,str(r.list_price)) for r in versions] == [
        ("Premium","Black","100.00"), ("Premium","Black","95.00"),
        ("Premium","Silver","95.00"), ("Standard","Silver","95.00"),
        ("Standard","Silver","90.00")]
    states = {r.order_id:r.status for r in spark.table("lab.ecommerce.latest_orders").collect()}
    assert {k:states[k] for k in [1001,1002,1003,1006,1009]} == {
        1001:"DELIVERED",1002:"CANCELLED",1003:"PAID",1006:"PAID",1009:"CREATED"}
    print("PASS: five product versions, late event, timestamp tie-break and mixed order states")
else:
    print("Continue source changes through Change 4 for final scenario checks.")
spark.sql("""SELECT selling_price, COUNT(*) AS orders, SUM(quantity) AS quantity,
    SUM(quantity*selling_price) AS booked_value FROM lab.ecommerce.latest_orders
    WHERE status IN ('PAID','DELIVERED') GROUP BY selling_price ORDER BY selling_price DESC""").show(truncate=False)

## Discussion

* CDC answers **what changed** in MySQL; SCD1 keeps current attributes; SCD2 keeps their history.
* Source sequence orders product changes. Business `event_time`, then `event_id`, orders order events.
* An order's actual selling price remains unchanged when today's product list price changes.
* Revisit notebook 1 for a new change, then run 2 → 3. For a separate dataset, change DB in all three.
* Snowflake will be a later exercise using these same raw batches, after this implementation is accepted.

References: [Iceberg 1.10 Spark writes](https://iceberg.apache.org/docs/1.10.0/spark-writes/),
[Hadoop catalog configuration](https://iceberg.apache.org/docs/1.10.0/spark-configuration/),
[MySQL Python binlog reader](https://github.com/julien-duponchelle/python-mysql-replication).